# Import Data

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ModuleNotFoundError: No module named 'xgboost'

# 1. Load Data

In [ ]:
raw_data = pd.read_csv("C:\Users\Probe\Desktop\jakarta_house.csv")

print("Data shape:", raw_data.shape)

print("\nPreview data:")
print(raw_data.head())

In [ ]:
# Hapus Kolom Index
raw_data = raw_data.drop(columns=['index'])
display(raw_data.head())

# 2. Explorary Data Analyst (EDA)

In [ ]:
data_explore = raw_data.copy()
data_explore.head()

In [ ]:
print("Data types dan cek null value")
print(data_explore.info())

print(f"\nJumlah District Unique: {data_explore["district"].nunique()}")

print("\nCity Unique")
print(data_explore['city'].unique())

In [ ]:
pd.set_option('display.float_format', lambda x: '%.2f' % x)
data_explore.describe().T

In [ ]:
# Ringkasan kualitas data
quality = pd.DataFrame({
    "dtype": data_explore.dtypes.astype(str),
    "missing": data_explore.isna().sum(),
    "missing_pct": (data_explore.isna().mean() * 100).round(2),
    "unique": data_explore.nunique(),
})
print(f"Jumlah baris: {len(data_explore):,}")
print(f"Jumlah duplikasi: {data_explore.duplicated().sum():,}")
display(quality)

In [ ]:
print("Cek anomali nilai yang = 0:")
for col in ["price", "land_area", "building_area", "bed_rooms", "bath_rooms"]:
    zero_count = (raw_data[col] == 0).sum()
    print(f"- {col} == 0: {zero_count} baris")
display(raw_data[raw_data["bed_rooms"] == 0][["district", "city", "land_area", "building_area", "bed_rooms", "bath_rooms", "price"]].head(10))

## 2.1 Distribusi Harga Rumah di Jakarta

In [ ]:
# Copy Data untuk kepentingan membuat plot
plot_data = data_explore.copy()
plot_data["price_miliar"] = plot_data["price"] / 1e9

In [ ]:
harga_miliar = plot_data["price_miliar"]
harga_mean = harga_miliar.mean()
harga_median = harga_miliar.median()

plt.figure(figsize=(10,6))

# Harga Rumah Jakarta < 50 M (Menghindari 1-2 data extreme)
sns.histplot(harga_miliar[harga_miliar < 50], bins=30, kde=True, color="navy")

plt.title("Distribusi Harga Rumah Jakarta")
plt.xlabel("Harga Rumah")
plt.ylabel("Jumlah Rumah")

plt.axvline(harga_mean, color="red", linestyle="--", linewidth=2, label=f"Mean: Rp{harga_mean:.2f} M")
plt.axvline(harga_median, color="green", linestyle="--", linewidth=2, label=f"Median: Rp{harga_median:.2f} M")

plt.legend()
plt.grid(True)
plt.show()

## 2.2 Analisis Harga Berdasarkan Lokasi & Luas Tanah

In [ ]:
category_counts = plot_data["city"].value_counts().sort_values(ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.barplot(x=category_counts.values, y=category_counts.index, ax=axes[0], color="steelblue")
axes[0].grid(axis="x", linestyle="--", alpha=0.5)
axes[0].set_title("Jumlah Data Rumah per Kota")
axes[0].set_xlabel("Jumlah Rumah")
axes[0].set_ylabel("Kota")

sns.barplot(
    data=plot_data,
    x="price_miliar",
    y="city",
    order=category_counts.index,
    estimator="median",
    errorbar=None,
    ax=axes[1]
)
axes[1].set_xticks(np.arange(0, 11, 1))
axes[1].grid(axis="x", linestyle="--", alpha=0.5)
axes[1].set_title("Median Harga Rumah per Kota")
axes[1].set_xlabel("Median Harga Rumah (Miliar)")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

display(
    plot_data.groupby("city")["price"]
    .agg(["count", "mean", "median", "min", "max"])
    .sort_values("mean", ascending=False)
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatterplot Luas Tanah < 2000 m2 dan Harga Rumah per Kota < Rp50 M
sns.scatterplot(data=plot_data, x="land_area", y="price_miliar", hue="city", alpha=0.35, s=22, ax=axes[0])
axes[0].set_xlim(0, 2000)
axes[0].set_ylim(0, 50)
axes[0].set_title("Hubungan Luas Tanah (< 2000 m2) vs Harga Rumah per Kota (< Rp50 M)")
axes[0].set_xlabel("Luas Tanah (m2)")
axes[0].set_ylabel("Harga Rumah (Miliar)")

# Memfilter distrik minimal 10 listing agar median tidak bias oleh 1-2 data extreme
district_counts = plot_data["district"].value_counts()
valid_districts = district_counts[district_counts >= 10].index

# Barplot
top_districts = (
    plot_data[plot_data["district"].isin(valid_districts)].groupby("district")["price_miliar"].median().sort_values(ascending=False).head(15)
)

sns.barplot(x=top_districts.values, y=top_districts.index, hue=top_districts.index, palette="viridis", legend=False, ax=axes[1])
axes[1].set_title("Top 15 District dengan Median Harga Total Rumah Tertinggi (>= 10 Listing)")
axes[1].set_xlabel("Median Harga Rumah (Miliar)")
axes[1].set_ylabel("District")

plt.tight_layout()
plt.show()

## 2.3 Rumah Termurah dan Termahal

In [ ]:
data_columns = ["district", "city", "bed_rooms", "bath_rooms", "land_area", "building_area", "price"]

print("10 RUMAH TERMURAH")
display(data_explore.nsmallest(10, "price")[data_columns])

print("10 RUMAH TERMAHAL")
display(data_explore.nlargest(10, "price")[data_columns])

## 2.4 Korelasi Antar Fitur & Deteksi Outlier

In [ ]:
# Korelasi fitur numerik terhadap target
numeric_columns = data_explore.select_dtypes(include="number").columns
correlation = data_explore[numeric_columns].corr(numeric_only=True)
plt.figure(figsize=(11, 8))
sns.heatmap(correlation, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Matriks Korelasi Variabel Numerik")
plt.tight_layout()
plt.show()

display(correlation["price"].drop("price").sort_values(key=abs, ascending=False).to_frame("Korelasi dengan Target"))

In [ ]:
# Pemeriksaan outlier berbasis IQR
outlier_summary = []
for column in numeric_columns:
    q1, q3 = data_explore[column].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    count = ((data_explore[column] < lower) | (data_explore[column] > upper)).sum()
    outlier_summary.append({"column": column, "IQR": round(iqr, 2), "Batas Bawah": round(lower, 2), "Batas Atas": round(upper, 2),"Jumlah Outlier": count, "Persentase Outlier": round(count / len(data_explore) * 100, 2)})

display(pd.DataFrame(outlier_summary).sort_values("Jumlah Outlier", ascending=False))

print("Insight awal:")
print(f"- Korelasi terkuat dengan harga: {correlation['price'].drop('price').abs().idxmax()}.")
print(f"- Kota dengan median harga tertinggi: {data_explore.groupby('city')['price'].median().idxmax()}.")
print(f"- Kolom dengan missing value: {quality[quality['missing'] > 0].index.tolist()}.")

# Data Cleaning

In [ ]:
clean_data = raw_data.copy()
display(clean_data.head())

data_awal = len(clean_data)
print(f"Jumlah data awal: {data_awal:,} baris")

In [ ]:
# Cek missing value
print("\nMissing values")
missing = clean_data.isnull().sum()
print(missing[missing > 0])

clean_data = clean_data.dropna(subset=["land_area", "building_area"])
print(f"Setelah Hapus Missing Value: {len(clean_data):,} baris")


In [ ]:
# Hapus Duplicate Values
print(f"Jumlah baris: {len(clean_data):,}")
print(f"Jumlah duplikasi: {clean_data.duplicated().sum():,}")

clean_data = clean_data.drop_duplicates()

print(f"Setelah Hapus Duplikat: {len(clean_data):,} baris")


In [ ]:
# Hapus Anomali properti (Tanpa Bedrooms dan Bathrooms)
clean_data = clean_data[(clean_data["bed_rooms"] > 0) & (clean_data["bath_rooms"] > 0)]
print(f"Setelah Hapus Anomali (Filtering): {len(clean_data):,} baris")

In [ ]:
# Cek Batas Harga Rumah Wajar
cek = clean_data[
    (clean_data["price"] >= 300000000) & ((clean_data["price"] / clean_data["land_area"]) >= 3500000)].sort_values(by="price")
cek.head(20)

In [ ]:
clean_data = clean_data[
    (clean_data["price"] >= 300000000) & ((clean_data["price"] / clean_data["land_area"]) >= 3500000)].sort_values(by="price")
print(f"Setelah Filter Harga & Harga/m2: {len(clean_data):,} baris")

In [ ]:
clean_data = clean_data.reset_index(drop=True)
print(f"Total Data Bersih: {len(clean_data):,} baris")

display(clean_data.head())

# Data Preprocessing

In [ ]:
encoded_data = clean_data.copy()

encoded_data.head()

In [ ]:
encoded_data = encoded_data[
    (encoded_data["bed_rooms"] >= 1) & (encoded_data["bed_rooms"] <= 15) &
    (encoded_data["bath_rooms"] >= 1) & (encoded_data["bath_rooms"] <= 15) &
    (encoded_data["land_area"] >= 15) & 
    (encoded_data["building_area"] >= 15)
]

for col in ["price", "land_area", "building_area"]:
    batas_atas = encoded_data[col].quantile(0.99)
    encoded_data = encoded_data[encoded_data[col] <= batas_atas]

harga_per_m2 = encoded_data["price"] / encoded_data["land_area"]
low_p, high_p = harga_per_m2.quantile([0.005, 0.995])
encoded_data = encoded_data[(harga_per_m2 >= low_p) & (harga_per_m2 <= high_p)]

print("Shape setelah outlier handling:", encoded_data.shape)

In [ ]:
# Encoding kolom teks 'city' dan 'district'
city_dummy = pd.get_dummies(encoded_data["city"], prefix="city", drop_first=True, dtype=int)
encoded_data = encoded_data.drop(columns=["city"])
encoded_data = pd.concat([encoded_data, city_dummy], axis=1)

district_freq = encoded_data["district"].value_counts() / len(encoded_data)
encoded_data["district_freq"] = encoded_data["district"].map(district_freq)
encoded_data = encoded_data.drop(columns=["district"])

In [ ]:
encoded_data.columns = [
    col.lower().replace('<', '_').replace(' ', '_') for col in encoded_data.columns
]

display(encoded_data)

In [ ]:
plt.figure(figsize=(16, 6))

heatmap = sns.heatmap(
    encoded_data.corr(),
    vmin=-1, vmax=1, annot=True,
    cmap="coolwarm"
)

heatmap.set_title("Correlation Heatmap", fontdict={"fontsize":12}, pad=12)

plt.show()

In [ ]:
final_data = encoded_data.copy()
final_data.head()

# Feature Engineering

In [ ]:
fe_data = final_data.copy()
display(fe_data.head())

# 1. Buat fitur baru dari rasio & total ruangan
fe_data["building_to_land_ratio"] = fe_data["building_area"] / fe_data["land_area"]
fe_data["total_rooms"] = fe_data["bed_rooms"] + fe_data["bath_rooms"]
fe_data["area_per_room"] = fe_data["building_area"] / fe_data["total_rooms"]

# 2. Log transformation untuk kolom yang miring/skewed
skewed_cols = ["land_area", "building_area"]
for col in skewed_cols:
    fe_data[f"log_{col}"] = np.log1p(fe_data[col])

fe_data = fe_data.replace([np.inf, -np.inf], np.nan).dropna()
print("Shape setelah feature engineering:", fe_data.shape)

# 3. Pisahkan target (y) dan fitur (X)
y = fe_data["price"]
X = fe_data.drop(columns=["price"] + skewed_cols)

print("Fitur yang dipakai:")
print(X.columns.tolist())

# 4. Split Train & Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train:", X_train.shape, "| Test:", X_test.shape)

# Experimentation

In [ ]:
models = {
    "LR": LinearRegression(),
    "RFR": RandomForestRegressor(n_jobs=-1),
    "XGBR": xgb.XGBRegressor()
}

In [ ]:
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print(f"{name} | R2: {r2:.4f} | MAE: ${mae:,.0f} | RMSE: ${rmse:,.0f}")

# Final Model

In [ ]:

rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE  : ${mae:,.0f}")
print(f"RMSE : ${rmse:,.0f}")
print(f"R²   : {r2:.4f}")